## Imports

In [176]:
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [177]:
df = pd.read_csv('../Clean_data/NPORS_2025_clean.csv')

In [179]:
df.head()

,RESPID,STRATUM,ECON1MOD,ECON1BMOD,COMTYPE2,UNITY,CRIMESAFE,GOVPROTCT,MOREGUNIMPACT,FIN_SIT,...,BIRTHPLACE,GENDER,ADULTS,VOTED2024,VOTEGEN_POST,INC_SDT1,CREGION,METRO,BASEWT,WEIGHT
0,1470,"10 = Remaining CBG, NOT 65%+ HS or less",Poor,Worse,Rural,Americans are divided when it comes to the mos...,Very safe,Sometimes laws to protect people from themselv...,More crime,Just meet my basic expenses,...,"U.S. - 50 states, District of Columbia",A woman,1,Yes,"Kamala Harris, the Democrat","$30,000 to less than $40,000",West,Metropolitan,0.802200,0.497038
1,2374,"7 = 75%+ Hispanic, 65%+ HS or less",Only fair,Worse,Urban,Americans are divided when it comes to the mos...,Somewhat safe,It's not the government's job to protect peopl...,No difference,Just meet my basic expenses,...,"U.S. - 50 states, District of Columbia",A man,4,Yes,"Kamala Harris, the Democrat","$70,000 to less than $100,000",West,Metropolitan,0.532494,0.307081
2,1177,"5 = 50%-74.99% Hispanic, 65%+ HS or less",Good,Better,Rural,Americans are divided when it comes to the mos...,Very safe,It's not the government's job to protect peopl...,Less crime,Just meet my basic expenses,...,"U.S. - 50 states, District of Columbia",A woman,2,Yes,"Donald Trump, the Republican","$30,000 to less than $40,000",West,Metropolitan,1.249642,0.646850
3,15459,"10 = Remaining CBG, NOT 65%+ HS or less",Only fair,About the same,Rural,Americans are divided when it comes to the mos...,Very safe,Sometimes laws to protect people from themselv...,No difference,Don't even have enough to meet basic expenses,...,"U.S. - 50 states, District of Columbia",A man,2,Yes,"Donald Trump, the Republican","Less than $30,000",Midwest,Non-metropolitan,1.604401,1.311245
4,9849,"9 = Remaining CBG, 65%+ HS or less",Good,Better,Suburban,Americans are divided when it comes to the mos...,Somewhat safe,It's not the government's job to protect peopl...,Less crime,Live comfortably,...,"U.S. - 50 states, District of Columbia",A man,4,Yes,"Donald Trump, the Republican","$150,000 or more",Northeast,Metropolitan,0.676351,0.241761


## Gender by Social Media Usage

In [182]:
sm_df = df.copy()
sm_df = sm_df.rename(columns=rename_dict)

rename_dict = {
    'SMUSE_FB': 'Facebook',
    'SMUSE_YT': 'YouTube',
    'SMUSE_X': 'X',
    'SMUSE_IG': 'Instagram',
    'SMUSE_SC': 'Snapchat',
    'SMUSE_WA': 'WhatsApp',
    'SMUSE_TT': 'TikTok',
    'SMUSE_RD': 'Reddit',
    'SMUSE_BSK': 'BeReal',
    'SMUSE_TH': 'Threads',
    'SMUSE_TS': 'Twitch'
}

social_cols = ['Facebook','YouTube','X','Instagram','Snapchat',
               'WhatsApp','TikTok','Reddit','BeReal','Threads','Twitch']

sm_df_melted = sm_df.melt(
    id_vars=['RESPID', 'GENDER'],
    value_vars=social_cols,
    var_name='platform',
    value_name='usage'
)

sm_df_melted = sm_df_melted[sm_df_melted['GENDER'].isin(['A man', 'A woman'])]

In [183]:
plot_df = (
    sm_df_melted.assign(using=sm_df_melted['usage'].eq('Yes, use this'))
    .groupby(['platform', 'GENDER'])['using']
    .mean()
    .reset_index(name='pct_using')
    .sort_values('pct_using', ascending=False)
)

plot_df['pct_using'] = plot_df['pct_using'].mul(100).round(2)

In [190]:
fig = px.bar(plot_df, x="platform", y="pct_using", color="GENDER")
fig.show()

In [ ]:
def get_links(df, col1, col2):
    return df.groupby([col1, col2]).size().reset_index(name='value').rename(columns={col1:'source', col2:'target'})

gender_df = df[df.GENDER.isin(['A man', 'A woman'])]  # Filter to men and women
gender_df = gender_df[gender_df.EDUCCAT.ne('Refused')]  # Filter out 'Refused' in EDUCCAT

df1 = get_links(gender_df, 'GENDER', 'EDUCCAT')
df2 = get_links(gender_df, 'EDUCCAT', 'VOTEGEN_POST')

all_links = pd.concat([df1, df2], axis=0)b

unique_nodes = pd.unique(all_links[['source', 'target']].values.ravel('K'))
node_map = {node: i for i, node in enumerate(unique_nodes)}


all_links['source'] = all_links['source'].map(node_map)
all_links['target'] = all_links['target'].map(node_map)

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=unique_nodes,
        color="blue"
    ),
    link=dict(
        source=all_links['source'],
        target=all_links['target'],
        value=all_links['value'],
        hovercolor=["midnightblue", "lightskyblue", "gold", "mediumturquoise", "lightgreen", "cyan"],
    ))])

fig.update_layout(title_text="Sankey Diagram: Flow from Gender to Education to Voting Results", font_size=10)
fig.show()


In [197]:
mapping = {
    "Less than $30,000": 15000,
    "$30,000 to less than $40,000": 35000,
    "$40,000 to less than $50,000": 45000,
    "$50,000 to less than $70,000": 60000,
    "$70,000 to less than $100,000": 85000,
    "$100,000 to less than $125,000": 112500,
    "$125,000 to less than $150,000": 137500,
    "$150,000 or more": 175000,
    "Refused/Web blank": np.nan
}
    
gender_df['income_numeric'] = gender_df['INC_SDT1'].apply(lambda x: mapping.get(x, np.nan))

In [215]:
fig = px.box(x=gender_df["GENDER"], y=gender_df["income_numeric"],color=gender_df["VOTEGEN_POST"])
fig.update_legends(title_text='Voting Post')
fig.update_xaxes(title_text='Gender')
fig.update_yaxes(title_text='Estimated Income')
fig.show()